In [1]:
!pip install -q pysam pyliftover
!apt-get install -qq -y samtools

In [2]:
!wget -q http://hgdownload.cse.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz
!gunzip -f hg19ToHg38.over.chain.gz

In [3]:
!wget -q --header="User-Agent: Mozilla/5.0" \
    -O GrafPkg.tar.gz \
    https://raw.githubusercontent.com/ncbi/graf/master/data/FP_SNPs.txt

!mkdir -p graf_extracted
!tar -xf GrafPkg.tar.gz -C graf_extracted
!ls -la graf_extracted/G1000FpGeno.bim

-rw-r--r-- 1 5383 584 276133 Sep 26  2017 graf_extracted/G1000FpGeno.bim


In [4]:
!head -3 graf_extracted/G1000FpGeno.bim
!wc -l graf_extracted/G1000FpGeno.bim

1	rs2887286	0	1156131	T	C
1	rs6685064	0	1211292	T	C
1	rs2840528	0	2283896	A	G
10000 graf_extracted/G1000FpGeno.bim


In [5]:
from pyliftover import LiftOver

lo = LiftOver("hg19ToHg38.over.chain")

with open("graf_extracted/G1000FpGeno.bim") as fin, \
     open("FP_SNPs_10k_GB38_twoAllelsFormat.tsv", "w") as fout:

    fout.write("#CHROM\tPOS\tID\tallele1\tallele2\n")
    total = converted = lost = 0

    for line in fin:
        parts = line.rstrip("\n").split("\t")
        if len(parts) != 6:
            continue
        total += 1
        chrom, rsid, _cm, pos37, a1, a2 = parts
        if chrom == "23":
            continue
        hit = lo.convert_coordinate("chr" + chrom, int(pos37))
        if not hit:
            lost += 1
            continue
        new_chrom, new_pos = hit[0][0], hit[0][1]
        fout.write(f"{new_chrom}\t{new_pos}\t{rsid}\t{a1}\t{a2}\n")
        converted += 1

print(f"total={total} converted={converted} lost={lost}")

total=10000 converted=10000 lost=0


In [6]:
!head -3 FP_SNPs_10k_GB38_twoAllelsFormat.tsv
!wc -l FP_SNPs_10k_GB38_twoAllelsFormat.tsv

#CHROM	POS	ID	allele1	allele2
chr1	1220751	rs2887286	T	C
chr1	1275912	rs6685064	T	C
10001 FP_SNPs_10k_GB38_twoAllelsFormat.tsv


In [7]:
import urllib.request

url = "https://api.gdc.cancer.gov/data/254f697d-310d-4d7d-a27b-27fbf767a834"
urllib.request.urlretrieve(url, "GRCh38.d1.vd1.fa.tar.gz")
print("downloaded")

downloaded


In [8]:
!mkdir -p ref
!tar -xzf GRCh38.d1.vd1.fa.tar.gz -C ref
!cd ref && samtools faidx GRCh38.d1.vd1.fa

!for c in 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 X Y M; do \
    samtools faidx ref/GRCh38.d1.vd1.fa chr$c > ref/chr$c.fa; \
    samtools faidx ref/chr$c.fa; \
 done

!ls ref/chr*.fa | head

ref/chr10.fa
ref/chr11.fa
ref/chr12.fa
ref/chr13.fa
ref/chr14.fa
ref/chr15.fa
ref/chr16.fa
ref/chr17.fa
ref/chr18.fa
ref/chr19.fa


In [9]:
%%writefile alleles_to_ref_alt.py
#!/usr/bin/env python3
"""
alleles_to_ref_alt.py

Convert a SNP table from
    #CHROM  POS  ID  allele1  allele2
to
    #CHROM  POS  ID  REF  ALT
by determining which of allele1/allele2 matches the reference base.

Reference genome is expected as per-chromosome FASTA files (chrN.fa)
with matching .fai indices, in a single directory.

Requires: pysam.
"""

import argparse
import logging
import os
import sys

import pysam


REQUIRED_HEADER = ["#CHROM", "POS", "ID", "allele1", "allele2"]
OUTPUT_HEADER = ["#CHROM", "POS", "ID", "REF", "ALT"]
VALID_NUCLEOTIDES = set("ACGTN")


def parse_args():
    parser = argparse.ArgumentParser(
        prog="alleles_to_ref_alt.py",
        description="Determine REF/ALT from allele1/allele2 against a reference FASTA.",
    )
    parser.add_argument("--input", "-i", required=True, help="Input TSV file.")
    parser.add_argument("--output", "-o", required=True, help="Output TSV file.")
    parser.add_argument("--reference-dir", "-r", required=True,
                        help="Directory with per-chromosome FASTA files (chrN.fa).")
    parser.add_argument("--log", "-l", default=None, help="Optional log file.")
    return parser.parse_args()


def setup_logging(log_path):
    logger = logging.getLogger("alleles_to_ref_alt")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s",
                            datefmt="%Y-%m-%d %H:%M:%S")
    console = logging.StreamHandler(sys.stdout)
    console.setFormatter(fmt)
    logger.addHandler(console)
    if log_path:
        d = os.path.dirname(os.path.abspath(log_path))
        if d and not os.path.isdir(d):
            os.makedirs(d, exist_ok=True)
        fh = logging.FileHandler(log_path, mode="w", encoding="utf-8")
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    return logger


def check_files(input_path, reference_dir, logger):
    if not os.path.isfile(input_path):
        logger.error("Input file not found: %s", input_path)
        sys.exit(1)
    if not os.path.isdir(reference_dir):
        logger.error("Reference directory not found: %s", reference_dir)
        sys.exit(1)


def validate_header(fh, logger):
    line = fh.readline()
    if not line:
        logger.error("Input file is empty.")
        sys.exit(1)
    header = line.rstrip("\r\n").split("\t")
    if header != REQUIRED_HEADER:
        logger.error("Unexpected header: %s", header)
        logger.error("Expected: %s", REQUIRED_HEADER)
        sys.exit(1)
    logger.info("Header validated.")


def get_fasta(chrom, reference_dir, cache, logger):
    """Open chrN.fa on demand and cache the Fastafile object."""
    if chrom in cache:
        return cache[chrom]

    path = os.path.join(reference_dir, f"{chrom}.fa")
    if not os.path.isfile(path):
        logger.warning("Reference FASTA not found for %s: %s", chrom, path)
        cache[chrom] = None
        return None
    if not os.path.isfile(path + ".fai"):
        logger.warning("FASTA index missing: %s.fai", path)
        cache[chrom] = None
        return None

    try:
        cache[chrom] = pysam.Fastafile(path)
        logger.info("Opened reference: %s", path)
    except Exception as exc:
        logger.warning("Failed to open %s: %s", path, exc)
        cache[chrom] = None
    return cache[chrom]


def convert_record(chrom, pos, allele1, allele2, fasta):
    """Return (ref, alt) or None if the record cannot be resolved."""
    ref_base = fasta.fetch(chrom, pos - 1, pos).upper()
    if not ref_base:
        return None
    a1, a2 = allele1.upper(), allele2.upper()
    if ref_base == a1 and ref_base != a2:
        return allele1, allele2
    if ref_base == a2 and ref_base != a1:
        return allele2, allele1
    return None


def main():
    args = parse_args()
    logger = setup_logging(args.log)

    logger.info("Started")
    logger.info("Input:         %s", args.input)
    logger.info("Output:        %s", args.output)
    logger.info("Reference dir: %s", args.reference_dir)

    check_files(args.input, args.reference_dir, logger)

    out_dir = os.path.dirname(os.path.abspath(args.output))
    if out_dir and not os.path.isdir(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    fasta_cache = {}
    total = written = skipped = 0

    with open(args.input, "r", encoding="utf-8", newline=None) as fin, \
         open(args.output, "w", encoding="utf-8", newline="\n") as fout:

        validate_header(fin, logger)
        fout.write("\t".join(OUTPUT_HEADER) + "\n")

        for lineno, line in enumerate(fin, start=2):
            line = line.rstrip("\r\n")
            if not line:
                continue
            total += 1
            fields = line.split("\t")

            if len(fields) != 5:
                logger.warning("Line %d: expected 5 fields, got %d", lineno, len(fields))
                skipped += 1
                continue

            chrom, pos_str, sid, a1, a2 = fields

            try:
                pos = int(pos_str)
            except ValueError:
                logger.warning("Line %d: POS not an integer (%s)", lineno, pos_str)
                skipped += 1
                continue

            if a1.upper() not in VALID_NUCLEOTIDES or a2.upper() not in VALID_NUCLEOTIDES:
                logger.warning("Line %d: invalid alleles %s/%s", lineno, a1, a2)
                skipped += 1
                continue

            fasta = get_fasta(chrom, args.reference_dir, fasta_cache, logger)
            if fasta is None:
                skipped += 1
                continue

            try:
                result = convert_record(chrom, pos, a1, a2, fasta)
            except Exception as exc:
                logger.warning("Line %d: fetch failed (%s)", lineno, exc)
                skipped += 1
                continue

            if result is None:
                logger.warning("Line %d: no reference match at %s:%d (a1=%s a2=%s)",
                               lineno, chrom, pos, a1, a2)
                skipped += 1
                continue

            ref, alt = result
            fout.write(f"{chrom}\t{pos}\t{sid}\t{ref}\t{alt}\n")
            written += 1

    for f in fasta_cache.values():
        if f is not None:
            f.close()

    logger.info("Finished: total=%d written=%d skipped=%d", total, written, skipped)


if __name__ == "__main__":
    main()

Overwriting alleles_to_ref_alt.py


In [10]:
!mkdir -p logs

!python3 alleles_to_ref_alt.py \
    --input  FP_SNPs_10k_GB38_twoAllelsFormat.tsv \
    --output FP_SNPs_10k_GB38_REF_ALT.tsv \
    --reference-dir ref \
    --log    logs/alleles_to_ref_alt.log

2026-10-08 18:44:14 [INFO] Started
2026-10-08 18:44:14 [INFO] Input:         FP_SNPs_10k_GB38_twoAllelsFormat.tsv
2026-10-08 18:44:14 [INFO] Output:        FP_SNPs_10k_GB38_REF_ALT.tsv
2026-10-08 18:44:14 [INFO] Reference dir: ref
2026-10-08 18:44:14 [INFO] Header validated.
2026-10-08 18:44:14 [INFO] Opened reference: ref/chr1.fa
2026-10-08 18:44:14 [WARNING] Line 80: invalid alleles 0/0
2026-10-08 18:44:14 [WARNING] Line 377: invalid alleles 0/0
2026-10-08 18:44:14 [WARNING] Line 420: no reference match at chr1:145899153 (a1=T a2=C)
2026-10-08 18:44:14 [WARNING] Line 438: invalid alleles 0/0
2026-10-08 18:44:14 [WARNING] Line 477: invalid alleles 0/0
2026-10-08 18:44:14 [WARNING] Line 699: invalid alleles 0/0
2026-10-08 18:44:14 [INFO] Opened reference: ref/chr2.fa
2026-10-08 18:44:14 [WARNING] Line 1238: invalid alleles 0/0
2026-10-08 18:44:14 [WARNING] Line 1283: invalid alleles 0/0
2026-10-08 18:44:14 [WARNING] Line 1644: invalid alleles 0/0
2026-10-08 18:44:14 [INFO] Opened refer

In [11]:
!head -3 FP_SNPs_10k_GB38_REF_ALT.tsv
!wc -l FP_SNPs_10k_GB38_REF_ALT.tsv
!head -10 logs/alleles_to_ref_alt.log
!tail -5 logs/alleles_to_ref_alt.log
!grep -c "invalid alleles 0/0" logs/alleles_to_ref_alt.log
!grep -c "no reference match" logs/alleles_to_ref_alt.log

#CHROM	POS	ID	REF	ALT
chr1	1220751	rs2887286	T	C
chr1	1275912	rs6685064	C	T
9917 FP_SNPs_10k_GB38_REF_ALT.tsv
2026-10-08 18:44:14 [INFO] Started
2026-10-08 18:44:14 [INFO] Input:         FP_SNPs_10k_GB38_twoAllelsFormat.tsv
2026-10-08 18:44:14 [INFO] Output:        FP_SNPs_10k_GB38_REF_ALT.tsv
2026-10-08 18:44:14 [INFO] Reference dir: ref
2026-10-08 18:44:14 [INFO] Header validated.
2026-10-08 18:44:14 [INFO] Opened reference: ref/chr1.fa
2026-10-08 18:44:14 [WARNING] Line 80: invalid alleles 0/0
2026-10-08 18:44:14 [WARNING] Line 377: invalid alleles 0/0
2026-10-08 18:44:14 [WARNING] Line 420: no reference match at chr1:145899153 (a1=T a2=C)
2026-10-08 18:44:14 [WARNING] Line 438: invalid alleles 0/0
2026-10-08 18:44:14 [INFO] Opened reference: ref/chr21.fa
2026-10-08 18:44:14 [WARNING] Line 9724: invalid alleles 0/0
2026-10-08 18:44:14 [WARNING] Line 9759: invalid alleles 0/0
2026-10-08 18:44:14 [INFO] Opened reference: ref/chr22.fa
2026-10-08 18:44:14 [INFO] Finished: total=10000 wr

In [12]:
with open("FP_SNPs_10k_GB38_REF_ALT.tsv") as f:
    f.readline()
    bad = [l for l in f if l.split("\t")[3] == l.split("\t")[4]]
    print("REF==ALT rows:", len(bad))

REF==ALT rows: 0
